In [ ]:
using Gridap        
using GridapGmsh
using Gridap.Geometry
using Gridap.TensorValues
using Plots
using LinearAlgebra
using  Gridap.Fields
using  Gridap.CellData
using  Gridap.ReferenceFEs  
using  Gridap.Fields
using Gridap.FESpaces
using NearestNeighbors

In [ ]:
using LinearAlgebra
using Distributions      
using Plots

In [ ]:
model = GmshDiscreteModel("UniaxialComp_Rock.msh")
writevtk(model,"UniaxialComp_Rock")

In [ ]:
const ν = 0.15
const E = 5964
const G = E/(2*(1+ν))

In [ ]:
const λ = (E*ν)/((1+ν)*(1-2*ν))
const μ = G
const κ = λ + μ

In [ ]:
D_x = 0.01
D_y = 0.01
G0_n = 7.0e-6
G0_s = 7.0e-6
τ = 1e-9
R = 0.01
Gc_n = 3.2
Gc_s = 10.7
Fr_Ang = 28
tFr_Ang = tand(Fr_Ang)

In [ ]:
θ = 0.0 

In [ ]:
using LinearAlgebra
using Random
using Plots

function generate_elliptical_points(N::Int, a, b, θ)
    
    θr = deg2rad(θ)

    R = [cos(θr) -sin(θr); sin(θr) cos(θr)]

   
    r = sqrt.(rand(N))     
    φ = 2.0 .* π .* rand(N)      
    x = r .* cos.(φ)
    y = r .* sin.(φ)

    pts = [x'; y']         

   
    scale = Diagonal([a, b])
    ellipse_pts = R' * (scale * pts)
point_objs = [Point(ellipse_pts[1, i], ellipse_pts[2, i]) for i in 1:N]
    return point_objs, φ
end


N = 70
a = 0.005 
b = 0.005 
ellipse_pts,Bond_orient = generate_elliptical_points(N, a, b, 0.0)
origin_point = Point(0.0, 0.0)
cod_final = ellipse_pts

r_x = [val[1] for val in cod_final ]


r_y = [val[2] for val in cod_final ]   
N = length(cod_final)
Dir_Vec = ones(N)


μ_0_pdf = 0.0
r_discr = norm.(cod_final)

C = [ (xi - xj)^2 for xi in r_discr, xj in r_discr ]  

ϵ = 0.5e-10

p0 = fill(1.0 / N, N)


x = [p[1] for p in cod_final]
y = [p[2] for p in cod_final]


scatter(x, y, legend=false,
        xlabel="x", ylabel="y")


In [ ]:
σ(ε)= λ*tr(ε)*one(ε) + 2*μ*ε 

In [ ]:
degree = 2
Ω = Triangulation(model)
dΩ = Measure(Ω,degree)

In [ ]:
labels = get_face_labeling(model)

In [ ]:
Gr = get_grid(model)
nodes = get_node_coordinates(Gr)
Nₑ, Nₙ = num_cells(model), num_nodes(model)
nodeCoordX, nodeCoordY = [nodes[i][1] for i in 1:Nₙ], [nodes[i][2] for i in 1:Nₙ]
elem = get_cell_node_ids(Gr)

In [ ]:
function G_kill(σ_eq_x_n, G0_eq)
G_kill_x = 0.5 .* G0_eq .* ((σ_eq_x_n  .- 1) .+ abs.(σ_eq_x_n  .- 1)  ) .* σ_eq_x_n ./ T
    return G_kill_x
end 

In [ ]:
function new_EnergyState(ψPlusPrev_in,ψhPos_in)
    ψPlus_in = ψhPos_in
    if ψPlus_in <= ψPlusPrev_in
        ψPlus_out = ψPlusPrev_in 
        damaged = false
    else
        ψPlus_out = ψPlus_in
        damaged = true
    end
    damaged, ψPlus_out   
end

In [ ]:
exp.(-Float64.(C) ./ ϵ)

In [ ]:

maxiter = 1
sinkhorn_iters = 1
tol = 1e-9
function sinkhorn_JKO_step(p_k_x, C, ϵ, τ, V_x, sinkhorn_iters, tol)

    N = length(p_k_x)
    p_k_x = Float64.(p_k_x)
   
    K =  exp.(-Float64.(C) ./ ϵ) 
    p_next_x = copy(p_k_x)
    q_x = copy(p_k_x)

    for t in 1:maxiter
     
        q_x = exp.(-2*(τ/ϵ) .* (2 ./ r_discr) .* V_x  .* D_x)

       
        p_tilde_x = p_k_x .* q_x

       
        u_x = ones(N)
        v_x = ones(N)
      
        small = 1e-16
        for _ in 1:sinkhorn_iters
            u_x = p_k_x ./ (K * v_x .+ small)
            v_x = p_tilde_x ./ (K' * u_x .+ small)

        end

        Π_x = Diagonal(u_x) * K * Diagonal(v_x)
        p_new_x = vec(sum(Π_x, dims=1))


        p_next_x = p_new_x

    end
    bar_w_nds_x = sum(p_next_x .* exp.(dot.(-cod_final, cod_final)))

    return bar_w_nds_x  ,  p_next_x
end

In [ ]:
function nonLocal_w_bar_fe(ε_field, p_0_nds_x,Gk_history_old) 
    
    ε_vals = evaluate(ε_field, x_S) 
    
    caches_x = [array_cache(ε_vals) for _ in 1:Threads.nthreads()]

   
    T_type = eltype(ε_vals[1]) 
    
   
    Gk_sum_x = zeros(T_type, Nₙ)
    Gk_count_x = zeros(Int, Nₙ)
    

    
    for iel in 1:Nₑ 
        cache_x = caches_x[Threads.threadid()]
        ElNdInd = elem[iel]
        
       
        val_ε_tensor = getindex!(cache_x, ε_vals, iel)
        
     
        tensor_ip = (sum(val_ε_tensor))/(length(val_ε_tensor)) 
        
        for node in ElNdInd
            Gk_sum_x[node] += tensor_ip
            Gk_count_x[node] += 1
        end
    end
    
    
    Gk_nds_x = Gk_sum_x ./ Gk_count_x
    

    

   
    w_bar_nds_x = zeros(Nₙ)
    p_new_x = zeros(N, Nₙ) 
    
    Gk_history_new = zeros(N, Nₙ)
    
    Threads.@threads for nd_id in 1:Nₙ
       
        G_tensor_nd = Gk_nds_x[nd_id]
        
   
        Gk_nd_Aniso_x1_temp = map(r -> dot(r, G_tensor_nd ⋅ r), cod_final ./ norm.(cod_final))
        
          Gk_nd_Aniso_x2_temp = map(n -> begin
    n_perp = VectorValue(-n[2], n[1]) 
    dot(n_perp, G_tensor_nd ⋅ n_perp)
            end, cod_final ./ norm.(cod_final)) 
        
        Gk_nd_Aniso_rn_temp = map(r -> begin
 n_perp = VectorValue(-r[2], r[1]) 
    dot(r, G_tensor_nd ⋅ n_perp) + dot(n_perp, G_tensor_nd ⋅ r)
end, cod_final ./ norm.(cod_final))
        
        Gk_nd_Aniso_x2 = Gk_nd_Aniso_x2_temp
        Gk_nd_Aniso_x1 = Gk_nd_Aniso_x1_temp 
        
        Gk_nd_Aniso_x1_nor = max.(Gk_nd_Aniso_x2_temp,Gk_nd_Aniso_x1_temp,0.0)
        Gk_nd_Aniso_x1_shear = 0.5 .* Gk_nd_Aniso_rn_temp
        Gk_nd_Aniso_x1_comp = min.(Gk_nd_Aniso_x2_temp + Gk_nd_Aniso_x1_temp,0.0)
        Gk_nd_Aniso_x1_cs = min.(Gk_nd_Aniso_rn_temp,0.0)
        
        Gk_nd_Eq = sqrt.( Gk_nd_Aniso_x1_nor .^2 ./ Gc_n.^2 +  Gk_nd_Aniso_x1_shear.^2 ./((Gc_s .- Gk_nd_Aniso_x1_nor .* tFr_Ang .* sign.(Gk_nd_Aniso_x1_shear) ).^2))
        mod_mix =  atan.(( max.(Gk_nd_Aniso_x2_temp, Gk_nd_Aniso_x1_temp) ) ./ abs.( ( (Gk_nd_Aniso_x1_shear)  ) .+ 1e-8) )
        #Gc_eq = sqrt.( (Gc_n.^2 .* (Gc_s .+ Gk_nd_Aniso_x2 .* tFr_Ang).^2) ./ (Gc_n.^2 .* (cos.(mod_mix)).^2 .+ (Gc_s .+ Gk_nd_Aniso_x2 .* tFr_Ang).^2 .* (sin.(mod_mix)).^2  ))
         G0_eq = sqrt.( (G0_n.^2 * G0_s.^2) ./ (G0_n.^2 .* (cos.(mod_mix)).^2 .+ G0_s.^2 .* (sin.(mod_mix)).^2  ))
        Gk_nd_Aniso = G_kill(Gk_nd_Eq , G0_eq)
        
        Gk_old_local = Gk_history_old[:, nd_id]
        Gk_max_local = max.(Gk_nd_Aniso, Gk_old_local)
        Gk_history_new[:, nd_id] .= Gk_max_local
        p_0_in_x = p_0_nds_x[:, nd_id]
        
        @inbounds w_bar_nds_x[nd_id], p_temp_x = sinkhorn_JKO_step(
            p_0_in_x, C, ϵ, τ,  Gk_max_local, sinkhorn_iters, tol
        )
        p_new_x[:, nd_id] .= p_temp_x
    end

    return FEFunction(Vphi, w_bar_nds_x .+ 1e-6), p_new_x, Gk_history_new 
end

In [ ]:
LoadTagId = get_tag_from_name(labels,"BottomEdge")
Γ_Load = BoundaryTriangulation(model,tags = LoadTagId)
dΓ_Load = Measure(Γ_Load,degree)
n_Γ_Load = get_normal_vector(Γ_Load)

LoadTagId2 = get_tag_from_name(labels,"TopEdge")
Γ_Load2 = BoundaryTriangulation(model,tags = LoadTagId2)
dΓ_Load2 = Measure(Γ_Load2,degree)
n_Γ_Load2 = get_normal_vector(Γ_Load2)


In [ ]:
reffe_Disp = ReferenceFE(lagrangian,VectorValue{2,Float64},1)
V0_Disp = TestFESpace(model,reffe_Disp;
          conformity=:H1,
          dirichlet_tags=["TopEdge","BottomEdge"],
          dirichlet_masks=[(false,true),(true,true)])

In [ ]:
reffephi  = ReferenceFE(lagrangian,Float64,1)
Vphi  = TestFESpace(model,reffephi;
          conformity=:H1)

In [ ]:
function step_disp(uApp,uh,p_0_nds_x,ϕ,Gk_history_new)
uApp1(x) = VectorValue(0.0,-uApp)
uApp2(x) = VectorValue(0.0,0.0)  
U_Disp = TrialFESpace(V0_Disp,[uApp1,uApp2])
ϕ,p_0_nds_x,Gk_history_new  = nonLocal_w_bar_fe(σ∘((ε(uh))) ,p_0_nds_x,Gk_history_new)
a(u,v) = ∫( ε(v) ⊙ ((σ∘((ε(u))))  .*(ϕ+1e-6)))dΩ
l(v) = 0.0 
op = AffineFEOperator(a,l,U_Disp,V0_Disp)
ls = LUSolver()
solver = LinearFESolver(ls)
uh = solve(solver,op)
    return uh, ϕ, p_0_nds_x, Gk_history_new 
end

In [ ]:
function project(q,model,dΩ,order)
  reffe = ReferenceFE(lagrangian,Float64,order)
  V = FESpace(model,reffe,conformity=:L2)
  a(u,v) = ∫( u*v )*dΩ
  l(v) = ∫( v*q )*dΩ
  op = AffineFEOperator(a,l,V,V)
  qh = solve(op)
  qh
end

In [ ]:
dΩ_ro = Measure(Ω,1)
x_S = get_cell_points(dΩ_ro)

In [ ]:
cd("SinkhornResults")
cd("MixedModeMshp75")
cd("N60abp01")

In [ ]:
uh = zero(V0_Disp)
Tmax = 0.7
delT = 0.0005
vApp = 1.0
count_n = 0
T = 0.0
Load = Float64[]
Displacement = Float64[]
push!(Load, 0.0)
push!(Displacement, 0.0)
Gk_history_new = zeros(N, Nₙ)
uh_prev = zero(V0_Disp)
uh_in_FE = uh
f_new = 1.0
ϕ_prev = interpolate_everywhere(f_new,Vphi)
ϕ = interpolate_everywhere(f_new,Vphi)
ϕ_x = interpolate_everywhere(f_new,Vphi)
ϕ_y = interpolate_everywhere(f_new,Vphi)
innerMax = 10
G_k_cell_x = CellState(0.0,dΩ_ro)
G_k_cell_y = CellState(0.0,dΩ_ro)
p0_nds_x = p0 .* ones(1, Nₙ)
p0_nds_y = p0 .* ones(1, Nₙ)
start_time = time()
while T <= Tmax
    count_n = count_n + 1

T = T + delT
uApp  = T*vApp
    print("\n Entering displacemtent step :", float(uApp))
 for inner = 1:innerMax
uh,ϕ,p0_nds_x,Gk_history_new =  step_disp(uApp,uh,p0_nds_x,ϕ,Gk_history_new)
e = uh - uh_in_FE

err = sqrt(sum( ∫( e⊙e )*dΩ ))
ϕ_prev = ϕ
uh_in_FE = uh
print("\n error = ",float(err))
        if err < 1e-8
            break 
        end  
    end
Node_Force = sum(∫( n_Γ_Load2 ⋅ (σ∘( (ε(uh))) ).*ϕ)dΓ_Load2)
push!(Load, abs(Node_Force[2]))
push!(Displacement, uApp)


if mod(count_n,10) == 0
writevtk(Ω,"results_NonLocal_$count_n",cellfields=["disp"=>uh,"phi"=>ϕ ])   
    end
    
    end
end_time = time()
elapsed_time = end_time - start_time